# Day 1: Ingest and Index Your Data

Welcome to the crash course!

In this course, you'll learn how to build intelligent systems that can understand and interact with your data.

We'll create a conversational agent that can answer questions about any GitHub repository — think of it as your personal AI assistant for documentation and code. If you know DeepWiki, it's something similar, but tailored to your GitHub repo.

For that, we need to:

1. Download and process data from the repo  
2. Put it inside a search engine  
3. Make the search engine available to our agent  

In the first half of the course, we will focus on data preparation. Today, we will do the first part: **downloading the data**.

## GitHub repo data

On the first day, we will learn how to download and process data from any GitHub repository. We will download the data as a zip archive, process all the text data from there, and make it available for ingesting it later into a search engine.

Think of this as preparing a meal — we need to gather and prep all our ingredients (the data) before we can cook (build our AI agent).

Today, we will deal with simple cases, when documents are not large. Tomorrow we will deal with more complex cases when documents are big and we also have code.

## Environment setup

First, let's prepare the environment. We need **Python 3.10 or higher**.

We will use **uv** as the package manager. If you don't have uv, install it:

```bash
pip install uv
```

This repo uses this layout:

- `course/` — reproduce all examples from this email course (this notebook lives here)
- `project/` — your own homework project

From the `course` directory, dependencies are already configured via uv:

```bash
uv init
uv add requests python-frontmatter
uv add --dev jupyter
```

This initializes a Python project and installs:

- **requests** — downloading data from GitHub  
- **python-frontmatter** — parsing structured metadata in markdown files  
- **jupyter** (dev) — development and experimentation only, not production code  

Start Jupyter from `course`:

```bash
uv run jupyter notebook
```

## Understanding frontmatter

We use a library for parsing **frontmatter** — a popular documentation format used with Jekyll, Hugo, Next.js, and similar tools.

It looks like this:

```markdown
---
title: "Getting Started with AI"
author: "John Doe"
date: "2024-01-15"
tags: ["ai", "machine-learning", "tutorial"]
difficulty: "beginner"
---

# Getting Started with AI

This is the main content of the document written in **Markdown**.
```

The section between the `---` markers contains YAML metadata; everything below is regular Markdown. We can extract structured information (title, tags, difficulty) along with the content.

In [ ]:
import frontmatter

# Example: create a small sample file to demonstrate (optional)
sample_md = """---
title: "Getting Started with AI"
author: "John Doe"
date: "2024-01-15"
tags: ["ai", "machine-learning", "tutorial"]
difficulty: "beginner"
---

# Getting Started with AI

This is the main content of the document written in **Markdown**.
"""

post = frontmatter.loads(sample_md)
print(post.metadata["title"])   # "Getting Started with AI"
print(post.metadata["tags"])    # ["ai", "machine-learning", "tutorial"]
print(post.content[:80], "...")  # markdown body without frontmatter

# All metadata + content as a dict
post.to_dict()

## Sample repositories

We'll use multiple repositories as a knowledge base:

- [DataTalksClub/faq](https://github.com/DataTalksClub/faq) (powers [datatalks.club/faq](https://datatalks.club/faq/)) — FAQ for DataTalks.Club courses  
- [evidentlyai/docs](https://github.com/evidentlyai/docs/) — docs for the Evidently AI library  

You can clone with git and process files, or **download the repo as a zip** — often easier for batch processing.

We can load the zip into memory with **requests** + **zipfile** + **io**, iterate `.md` and `.mdx` files, and collect results in a list.

## Working with zip archives

Plan:

1. Use `requests` to download the zip from GitHub  
2. Open the archive with `zipfile` and `io.BytesIO`  
3. Iterate over `.md` / `.mdx` files  
4. Parse with frontmatter and append to a list  

GitHub zip URL format:

`https://codeload.github.com/{owner}/{repo}/zip/refs/heads/main`

In [ ]:
import io
import zipfile

import frontmatter
import requests

url = "https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main"
resp = requests.get(url)
resp.raise_for_status()

repository_data = []
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()
    if not filename.endswith(".md"):
        continue
    with zf.open(file_info) as f_in:
        content = f_in.read().decode("utf-8", errors="ignore")
        post = frontmatter.loads(content)
        data = post.to_dict()
        data["filename"] = filename
        repository_data.append(data)

zf.close()
len(repository_data), repository_data[1] if len(repository_data) > 1 else repository_data[0]

For Evidently docs, include **`.mdx`** (React markdown):

```python
if not (filename.endswith('.md') or filename.endswith('.mdx')):
    continue
```

## Complete implementation

Reusable function to download and parse all markdown files from a GitHub repository.

In [ ]:
import io
import zipfile

import frontmatter
import requests


def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = "https://codeload.github.com"
    url = f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)

    if resp.status_code != 200:
        raise RuntimeError(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith(".md") or filename_lower.endswith(".mdx")):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)
                data = post.to_dict()
                data["filename"] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    zf.close()
    return repository_data

In [ ]:
dtc_faq = read_repo_data("DataTalksClub", "faq")
evidently_docs = read_repo_data("evidentlyai", "docs")

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

## Data processing considerations

- **FAQ**: Small records — often fine to index as-is.  
- **Evidently docs**: Large documents — need **chunking** (splitting into smaller pieces) for search relevance, model context limits, and performance.

Chunking is covered in the next lesson.

## Project and homework

Work in the separate **`project/`** folder (its own uv project with the same dependencies).

1. Create or use the uv project in `project/`  
2. Pick a GitHub repo with documentation (preferably `.md` files)  
3. Download and parse it using the techniques above  
